# Ocean Optics — independent re-check of the numbers on the site

This notebook recomputes the key numbers **from scratch with short, independent code** and compares them with `data/verified.json` (written by `build/verify_numbers.py`). Every comparison prints `OK` or fails the notebook. Sources of the formulas: lecture slides (numbers cited) and the Ocean Optics Web Book (CC BY 2.0). Mie spheres and the sphere package model are illustrations, not data.

In [1]:
import json, math, numpy as np
from scipy import integrate, optimize
from scipy.special import spherical_jn, spherical_yn
V = json.load(open('../data/verified.json'))
def ok(name, a, b, tol):
    rel = abs(a-b)/max(abs(a), abs(b), 1e-300); assert rel <= tol, (name, a, b, rel); print('OK ', name, a, b)
print(V['n_checks'], 'checks in verified.json, passed:', V['n_ok'])

91 checks in verified.json, passed: 91.0


## 1. Pure-water scattering (slides 20, 23; cos²θ reading)

In [2]:
def beta(th, lam, S, A, e, d): return A*(lam/500)**e*(1+0.3*S/37)*1e-4*(1+np.cos(th)**2*(1-d)/(1+d))
for key, (A, e, d) in {'morel': (1.38, -4.32, .09), 'buiteveld': (1.29, -4.14, .051)}.items():
    for lam in (450, 500, 550):
        b = integrate.quad(lambda t: 2*math.pi*beta(t, lam, 37, A, e, d)*math.sin(t), 0, math.pi)[0]
        ok(f'b_w {key} {lam}', b, V['pure_water_bw'][f'{key}_{lam}_S37']['b_w'], 1e-8)
ok('ratio 450/550 (-4.32)', (550/450)**4.32, V['ratio_450_550_morel'], 1e-12)
ok('b_bw/b_w = 1/2', integrate.quad(lambda t: math.sin(t)*beta(t,500,37,1.38,-4.32,.09), math.pi/2, math.pi)[0] / integrate.quad(lambda t: math.sin(t)*beta(t,500,37,1.38,-4.32,.09), 0, math.pi)[0], 0.5, 1e-9)

OK  b_w morel 450 0.004542900146429388 0.004542900146429388
OK  b_w morel 500 0.002881780058942822 0.002881780058942822
OK  b_w morel 550 0.001909169255130458 0.001909169255130458
OK  b_w buiteveld 450 0.004240827650256788 0.004240827650256788
OK  b_w buiteveld 500 0.0027416664142147136 0.0027416664142147136
OK  b_w buiteveld 550 0.0018477741850454203 0.0018477741850454203
OK  ratio 450/550 (-4.32) 2.3795167108528372 2.3795167108528372
OK  b_bw/b_w = 1/2 0.5000000000000001 0.5


## 2. Exponential absorption (slides 35, 41) and power law (slide 49)

In [3]:
ok('CDOM 443/412', math.exp(-0.014*31), V['cdom_ratio_443_from_412_S014'], 1e-12)
ok('CDOM 350/440', math.exp(0.014*90), V['cdom_ratio_350_over_440_S014'], 1e-12)
ok('NAP 350/440', math.exp(0.011*90), V['nap_ratio_350_over_440_S011'], 1e-12)
ok('bp 440/550, n=-1', (440/550)**-1, V['bp_440_over_550']['-1'], 1e-12)
ok('bp 440/550, n=-3', (440/550)**-3, V['bp_440_over_550']['-3'], 1e-12)

OK  CDOM 443/412 0.6479122554853506 0.6479122554853506
OK  CDOM 350/440 3.5254214873653824 3.5254214873653824
OK  NAP 350/440 2.691234472349262 2.691234472349262
OK  bp 440/550, n=-1 1.25 1.25
OK  bp 440/550, n=-3 1.9531249999999998 1.9531249999999998


## 3. Beer–Lambert (ODE), euphotic and Secchi depth (textbook)

In [4]:
sol = integrate.solve_ivp(lambda x, I: -1.0*I, [0, 0.25], [1.0], rtol=1e-11, atol=1e-13)
ok('T(c=1,L=0.25)', float(sol.y[0,-1]), V['beer_lambert_T']['c1.0_L0.25'], 1e-7)
ok('c from T=0.9, L=0.25', -math.log(0.9)/0.25, V['c_from_T_25cm']['0.9'], 1e-12)
ok('z1% Kd=0.04', optimize.brentq(lambda z: math.exp(-0.04*z)-0.01, 0, 1e4), V['euphotic_1pct']['0.04']['z1pct'], 1e-8)
ok('Secchi Kd=.1,c=.2', 8/(0.1+0.2), V['secchi_examples']['Kd0.1_c0.2'], 1e-12)

OK  T(c=1,L=0.25) 0.7788007830717946 0.7788007830714049
OK  c from T=0.9, L=0.25 0.4214420626313051 0.4214420626313051
OK  z1% Kd=0.04 115.12925464970229 115.12925464970228
OK  Secchi Kd=.1,c=.2 26.666666666666664 26.666666666666664


## 4. Package effect of a sphere (slide 33 concept; ray-path integral)

In [5]:
def q_ray(rho):
    R=1.0; a=rho/(2*R)
    f=lambda b:(1-math.exp(-a*2*math.sqrt(max(R*R-b*b,0))))*2*math.pi*b
    return integrate.quad(f,0,R,limit=200)[0]/(a*4/3*math.pi*R**3)
for r in ('0.2','1.0','5.0'): ok('Q*_a rho='+r, q_ray(float(r)), V['package_Qstar'][r], 1e-3)

OK  Q*_a rho=0.2 0.9288388850918345 0.9288388850918228
OK  Q*_a rho=1.0 0.7072766470286571 0.707276647028654
OK  Q*_a rho=5.0 0.27697026436786787 0.27697026436786826


## 5. Mie ensemble: Rayleigh limit and Junge exponent n = −(ξ−3) (slides 49, 68)

In [6]:
def coeffs(m, x):
    nmax=int(x+4*x**(1/3)+2); n=np.arange(1,nmax+1); mx=m*x
    jx,jpx=spherical_jn(n,x),spherical_jn(n,x,derivative=True); yx,ypx=spherical_yn(n,x),spherical_yn(n,x,derivative=True)
    jm,jpm=spherical_jn(n,mx),spherical_jn(n,mx,derivative=True)
    psi,dpsi=x*jx,jx+x*jpx; psim,dpsim=mx*jm,jm+mx*jpm; xi=x*(jx+1j*yx); dxi=(jx+1j*yx)+x*(jpx+1j*ypx)
    a=(m*psim*dpsi-psi*dpsim)/(m*psim*dxi-xi*dpsim); b=(psim*dpsi-m*psi*dpsim)/(psim*dxi-m*xi*dpsim); return n,a,b
def qsca(m,x):
    n,a,b=coeffs(m,x); return 2/x**2*np.sum((2*n+1)*(abs(a)**2+abs(b)**2))
xr=0.02; ok('Rayleigh Qsca m=1.15', qsca(1.15,xr), 8/3*xr**4*((1.15**2-1)/(1.15**2+2))**2, 5e-3)
def bp(xi, lam):
    D=np.logspace(-1,2,140); x=math.pi*D*1.34/(lam/1000); w=D*np.gradient(np.log(D))
    return sum(D_**-xi*math.pi*(D_/2)**2*qsca(1.05,x_)*w_ for D_,x_,w_ in zip(D,x,w))
n_bp = math.log(bp(4.0,440)/bp(4.0,660))/math.log(440/660)
ok('Junge xi=4 exponent', n_bp, V['junge_ensemble_m105']['4.0']['n_bp'], 5e-3); print('n =', n_bp, 'expected -(xi-3) = -1')

OK  Rayleigh Qsca m=1.15 4.019533596664591e-09 4.019927065445764e-09


OK  Junge xi=4 exponent -1.0070786600394874 -1.0070786600394888
n = -1.0070786600394874 expected -(xi-3) = -1
